# SQL: загрузка таблиц в базу данных и создание VIEW

## Что делает этот ноутбук
1. Загружает все очищенные CSV-таблицы в SQLite базу `moscow_realty.db`
2. Создаёт VIEW для аналитических запросов (payback period, наценка новостроек, сверка с официальными данными, джойн с метро)


In [1]:
import sqlite3
import pandas as pd
import os
from pathlib import Path

# Указание Директории
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

PROJECT_ROOT = Path(os.getcwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SQL_DIR = PROJECT_ROOT / 'sql'
DB_PATH = PROCESSED_DIR / 'moscow_realty.db'

print(f"Корень проекта: {PROJECT_ROOT}")
print(f"База данных: {DB_PATH}")
print(f"База существует: {DB_PATH.exists()}")

Корень проекта: D:\Moscow_rent_analysis
База данных: D:\Moscow_rent_analysis\data\processed\moscow_realty.db
База существует: True


## Шаг 1. Загрузка очищенных таблиц в SQLite

Загружаем все пять таблиц из `data/processed/` в базу данных.
`if_exists='replace'` — если таблица уже есть, перезаписываем (актуально при повторном запуске после изменений в данных).

In [2]:
# Таблицы для загрузки: (имя файла, имя таблицы в БД)
tables = [
    ('clean_secondary_market.csv', 'secondary_market'),
    ('clean_rentals.csv',          'rentals'),
    ('clean_new_builds.csv',       'new_builds'),
    ('clean_district_prices_monthly.csv', 'district_prices_monthly'),
]

# metro_stations берём из raw, т.к. не чистили отдельно
metro_path = PROJECT_ROOT / 'data' / 'raw' / 'metro_stations.csv'

conn = sqlite3.connect(str(DB_PATH))

# Загружаем основные таблицы
for filename, table_name in tables:
    df = pd.read_csv(PROCESSED_DIR / filename)
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Загружено: {table_name} ({len(df):,} строк)")

# Загружаем metro_stations отдельно
df_metro = pd.read_csv(metro_path)
df_metro.to_sql('metro_stations', conn, if_exists='replace', index=False)
print(f"Загружено: metro_stations ({len(df_metro):,} строк)")

conn.commit()
print("\nВсе таблицы успешно загружены в БД")

Загружено: secondary_market (50,000 строк)
Загружено: rentals (20,000 строк)
Загружено: new_builds (8,000 строк)


Загружено: district_prices_monthly (9,804 строк)
Загружено: metro_stations (104 строк)

Все таблицы успешно загружены в БД


## Шаг 2. Проверка загруженных таблиц

Убеждаемся, что все таблицы на месте и количество строк совпадает с ожидаемым.

In [3]:
# Список всех таблиц в базе
tables_in_db = pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name",
    conn
)
print("Объекты в базе данных:")
print(tables_in_db.to_string(index=False))

print()

# Количество строк в каждой таблице
all_tables = ['secondary_market', 'rentals', 'new_builds', 'district_prices_monthly', 'metro_stations']
for t in all_tables:
    count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn).iloc[0, 0]
    print(f"{t}: {count:,} строк")

Объекты в базе данных:
                    name  type
 district_prices_monthly table
          metro_stations table
              new_builds table
                 rentals table
        secondary_market table
     v_new_build_premium  view
      v_new_builds_metro  view
        v_official_check  view
        v_payback_period  view
         v_rentals_metro  view
v_secondary_market_metro  view

secondary_market: 50,000 строк
rentals: 20,000 строк
new_builds: 8,000 строк
district_prices_monthly: 9,804 строк
metro_stations: 104 строк


## Шаг 3. Создание VIEW для аналитики

VIEW — это сохранённые запросы внутри базы данных. Они не дублируют данные на диске,
а пересчитываются из исходных таблиц при каждом обращении. Это позволяет:
- Просматривать результаты прямо в PyCharm/DBeaver как обычные таблицы
- Обращаться к ним из Python одной строкой: `pd.read_sql_query('SELECT * FROM v_payback_period', conn)`

Сначала удаляем VIEW если они уже существуют — на случай повторного запуска ноутбука.

In [4]:
# Удаляем старые VIEW перед пересозданием
views_to_drop = [
    'v_payback_period', 'v_new_build_premium', 'v_official_check',
    'v_rentals_metro', 'v_secondary_market_metro', 'v_new_builds_metro',
]

for view in views_to_drop:
    conn.execute(f"DROP VIEW IF EXISTS {view}")
    print(f"Удалён (если был): {view}")

conn.commit()
print("\nГотово к созданию новых VIEW")

Удалён (если был): v_payback_period
Удалён (если был): v_new_build_premium
Удалён (если был): v_official_check
Удалён (если был): v_rentals_metro
Удалён (если был): v_secondary_market_metro
Удалён (если был): v_new_builds_metro

Готово к созданию новых VIEW


In [5]:
# Создаём VIEW из SQL-файлов
sql_files = [
    (SQL_DIR / 'payback_period.sql',     'v_payback_period'),
    (SQL_DIR / 'new_build_premium.sql',  'v_new_build_premium'),
    (SQL_DIR / 'official_check.sql',     'v_official_check'),
    (SQL_DIR / 'metro_rentals_join.sql',          'v_rentals_metro'),
    (SQL_DIR / 'metro_secondary_market_join.sql', 'v_secondary_market_metro'),
    (SQL_DIR / 'metro_new_builds_join.sql',       'v_new_builds_metro'),
]

for sql_path, view_name in sql_files:
    with open(sql_path, encoding='utf-8') as f:
        sql = f.read()
    conn.execute(sql)
    conn.commit()
    print(f"Создан VIEW: {view_name}")

print("\nВсе VIEW успешно созданы")

Создан VIEW: v_payback_period
Создан VIEW: v_new_build_premium
Создан VIEW: v_official_check
Создан VIEW: v_rentals_metro
Создан VIEW: v_secondary_market_metro
Создан VIEW: v_new_builds_metro

Все VIEW успешно созданы


## Шаг 4. Быстрая проверка VIEW — смотрим первые строки

In [6]:
# Переоткрываем соединение для проверки
conn = sqlite3.connect(str(DB_PATH))

# Проверяем payback_period
df_payback = pd.read_sql_query("SELECT * FROM v_payback_period LIMIT 10", conn)
print("=== v_payback_period (топ-10 по выгодности покупки) ===")
print(df_payback.to_string(index=False))

=== v_payback_period (топ-10 по выгодности покупки) ===
           district okrug  avg_price_sqm  avg_rent_sqm  payback_years   recommendation
     Molzhaninovsky   SAO  113362.402089    637.993056           14.8 выгодно покупать
            Severny  SVAO  121340.487805    665.474820           15.2       нейтрально
    Yuzhnoye Butovo YuZAO  146251.058201    801.135484           15.2       нейтрально
            Vnukovo   ZAO  141744.881890    774.374233           15.3       нейтрально
          Solntsevo   ZAO  164367.821782    889.924419           15.4       нейтрально
             Mitino  SZAO  194027.150538   1041.613095           15.5       нейтрально
   Novo-Peredelkino   ZAO  150198.138298    806.388060           15.5       нейтрально
Chertanovo Yuzhnoye  YuAO  185671.538462    987.554054           15.7       нейтрально
         Tsaritsyno  YuAO  147479.277108    778.372414           15.8       нейтрально
          Veshnyaki   VAO  139651.554404    738.015504           15.8     

In [7]:
# Проверяем new_build_premium — наценка новостроек vs вторичка
df_premium = pd.read_sql_query("SELECT * FROM v_new_build_premium LIMIT 10", conn)
print("=== v_new_build_premium (топ-10 районов по наценке новостроек) ===")
print(df_premium.to_string(index=False))

=== v_new_build_premium (топ-10 районов по наценке новостроек) ===
                   district  avg_secondary  avg_newbuild  premium_pct
             Beskudnikovsky  150227.671233 330024.242424        119.7
     Pokrovskoye-Streshnevo  199965.789474 422769.387755        111.4
                   Khovrino  143824.409449 276681.081081         92.4
           Novo-Peredelkino  150198.138298 274998.809524         83.1
     Ochakovo-Matveyevskoye  204158.536585 372880.000000         82.6
           Yuzhnoye Tushino  184025.729443 328280.434783         78.4
Orekhovo-Borisovo Severnoye  135570.649351 237209.302326         75.0
                  Veshnyaki  139651.554404 244240.449438         74.9
              Akademichesky  255937.469586 440106.250000         72.0
      Khoroshyovo-Mnyovniki  228640.000000 382621.428571         67.3


In [8]:
# Проверяем official_check — сверка наших расчётов с официальными данными
df_check = pd.read_sql_query("SELECT * FROM v_official_check LIMIT 10", conn)
print("=== v_official_check (сверка с district_prices_monthly) ===")
print(df_check.to_string(index=False))

=== v_official_check (сверка с district_prices_monthly) ===
year_month       district  n_listings  our_avg_price  official_price  difference  difference_pct
2020-01-01       Aeroport           3       154400.0          220700    -66300.0           -30.0
2020-01-01  Akademichesky           4       185400.0          282900    -97500.0           -34.5
2020-01-01    Alekseevsky           9       157422.0          227900    -70478.0           -30.9
2020-01-01    Altufyevsky           4        96925.0          173500    -76575.0           -44.1
2020-01-01          Arbat           4       374250.0          480000   -105750.0           -22.0
2020-01-01   Babushkinsky           3       118300.0          186300    -68000.0           -36.5
2020-01-01       Basmanny           5       245700.0          380000   -134300.0           -35.3
2020-01-01        Begovoy           2       187100.0          246600    -59500.0           -24.1
2020-01-01 Beskudnikovsky           5       114020.0          17820

In [9]:
# Проверяем VIEW с метро-данными — сколько строк не нашли пару по названию станции
for view in ['v_rentals_metro', 'v_secondary_market_metro', 'v_new_builds_metro']:
    check = pd.read_sql_query(
        f"SELECT COUNT(*) AS total, SUM(year_opened IS NULL) AS missing_year_opened FROM {view}",
        conn
    ).iloc[0]
    print(f"{view}: {check['total']} строк, без year_opened: {check['missing_year_opened']}")

conn.close()

v_rentals_metro: 20000 строк, без year_opened: 0
v_secondary_market_metro: 50000 строк, без year_opened: 0
v_new_builds_metro: 8000 строк, без year_opened: 0


## Итоговые выводы по всем сегментам

Сводим воедино метрики, посчитанные по отдельности в `data_clean_rentals.ipynb`, `data_clean_secondary_market.ipynb`, `data_clean_new_builds.ipynb`, и добавляем агрегаты по SQL-представлениям `v_payback_period` и `v_new_build_premium`.

In [10]:
conn = sqlite3.connect(str(DB_PATH))

summary_rows = []
for table, price_col, label in [
    ('rentals', 'rent_per_sqm', 'Аренда'),
    ('secondary_market', 'price_per_sqm', 'Вторичка'),
    ('new_builds', 'price_per_sqm', 'Новостройки'),
]:
    df = pd.read_sql_query(f"SELECT {price_col}, is_premium, is_new_moscow FROM {table}", conn)
    # SQLite хранит булевы колонки как INTEGER (0/1), pandas читает их как int64 —
    # приводим к bool явно, иначе побитовое `~` даст не то, что ожидается
    df['is_premium'] = df['is_premium'].astype(bool)
    df['is_new_moscow'] = df['is_new_moscow'].astype(bool)

    old_median = df[~df['is_new_moscow']][price_col].median()
    new_median = df[df['is_new_moscow']][price_col].median()
    summary_rows.append({
        'Сегмент': label,
        'Медиана Старая Москва': round(old_median),
        'Медиана Новая Москва': round(new_median),
        'Разница, раз': round(old_median / new_median, 1),
        'Доля премиум, %': round(df['is_premium'].mean() * 100, 1),
        'Доля Новой Москвы, %': round(df['is_new_moscow'].mean() * 100, 1),
    })

df_summary = pd.DataFrame(summary_rows)
print("=== Сводка по сегментам: Старая vs Новая Москва, премиум-сегмент ===")
print(df_summary.to_string(index=False))

print("\n=== Распределение окупаемости по районам (v_payback_period) ===")
df_payback_dist = pd.read_sql_query(
    "SELECT recommendation, COUNT(*) AS n_districts, ROUND(AVG(payback_years),1) AS avg_years "
    "FROM v_payback_period GROUP BY recommendation ORDER BY avg_years",
    conn
)
print(df_payback_dist.to_string(index=False))

print("\n=== Наценка новостроек над вторичкой (v_new_build_premium) ===")
df_premium_stats = pd.read_sql_query(
    "SELECT ROUND(AVG(premium_pct),1) AS avg_premium_pct, "
    "ROUND(MIN(premium_pct),1) AS min_premium_pct, "
    "ROUND(MAX(premium_pct),1) AS max_premium_pct "
    "FROM v_new_build_premium",
    conn
)
print(df_premium_stats.to_string(index=False))

conn.close()

=== Сводка по сегментам: Старая vs Новая Москва, премиум-сегмент ===
    Сегмент  Медиана Старая Москва  Медиана Новая Москва  Разница, раз  Доля премиум, %  Доля Новой Москвы, %
     Аренда                    922                   418           2.2              5.1                  17.0
   Вторичка                 188500                 74800           2.5              5.4                  17.0
Новостройки                 258150                114650           2.3              3.8                  20.5

=== Распределение окупаемости по районам (v_payback_period) ===
  recommendation  n_districts  avg_years
выгодно покупать            1       14.8
      нейтрально          106       17.2

=== Наценка новостроек над вторичкой (v_new_build_premium) ===
 avg_premium_pct  min_premium_pct  max_premium_pct
            36.9            -10.4            119.7


**Разница старая vs Новая Москва устойчива по всем трём сегментам** — независимо от того, снимаешь, покупаешь на вторичке или в новостройке, жильё в старых границах города дороже в 2.2–2.5 раза. Сильнее всего разрыв на вторичке (2.5x), слабее в аренде (2.2x) — премиум-жильё (элитный ЦАО) сильнее тянет вверх именно цены продажи, а не аренды.

**Доля премиум-сегмента ниже всего у новостроек (3.8% против ~5% у аренды и вторички)** — рынок новостроек за рассматриваемый период (2020-2026) меньше ориентирован на элитный сегмент, что согласуется с выводом из `data_clean_new_builds.ipynb`.

**Доля Новой Москвы выше именно у новостроек (20.5% против 17% у аренды и вторички)** — застройщики активнее строят именно в новых районах, чем распределены уже существующие вторичные и арендные объекты.

**Окупаемость покупки через аренду (`v_payback_period`) почти везде нейтральная**: только 1 район из 107 попадает в категорию "выгодно покупать" (< 15 лет), остальные 106 — "нейтрально" (в среднем 17.2 года). Ни один район не попал в "выгодно снимать" (> 25 лет) — по чистой экономике покупка vs аренда в Москве почти нигде явно не проигрывает, но и явного перевеса в пользу покупки почти нет.

**Наценка новостроек над вторичкой в среднем +36.9%, но разброс огромный** — от -10.4% (в отдельном районе новостройка дешевле аналогичной вторички) до +119.7%. Говорить о единой "наценке новостроек" по Москве некорректно — решение сильно зависит от конкретного района.

> **Пересчитано 2026-09-05.** Все числа в этом разделе получены уже после исправления методики премиум-сегмента (пункт 8.5.3: порог IQR считается внутри года, а не один на весь период 2020-2026). До исправления здесь стояли: доля премиума у новостроек 3.6%, средняя окупаемость 17.1 года, наценка новостроек +37.3% при минимуме -10.6%. Сдвиги небольшие, потому что оба SQL-представления премиум *исключают*, а не изучают — смена метки у 1-2% объявлений почти не двигает средние по районам. Разбор самой ошибки — в трёх ноутбуках очистки и в разделе 8.5 `deep_analysis.ipynb`.


## Шаг 5. Закрываем соединение

In [11]:
conn.close()
print("Соединение с БД закрыто")
print(f"\nИтог: база данных {DB_PATH.name} содержит:")
print("  5 таблиц: secondary_market, rentals, new_builds, district_prices_monthly, metro_stations")
print("  6 VIEW:   v_payback_period, v_new_build_premium, v_official_check,")
print("            v_rentals_metro, v_secondary_market_metro, v_new_builds_metro")
print("\nДля аналитики открывай соединение заново в каждом ноутбуке:")
print("  conn = sqlite3.connect('data/processed/moscow_realty.db')")
print("  df = pd.read_sql_query('SELECT * FROM v_payback_period', conn)")

Соединение с БД закрыто

Итог: база данных moscow_realty.db содержит:
  5 таблиц: secondary_market, rentals, new_builds, district_prices_monthly, metro_stations
  6 VIEW:   v_payback_period, v_new_build_premium, v_official_check,
            v_rentals_metro, v_secondary_market_metro, v_new_builds_metro

Для аналитики открывай соединение заново в каждом ноутбуке:
  conn = sqlite3.connect('data/processed/moscow_realty.db')
  df = pd.read_sql_query('SELECT * FROM v_payback_period', conn)
